# NB12 — Mini Project and Assessment

**Module 17: HPC, Parallel Computing, and Rust**

---

## Mini Project: Pairwise Sequence Identity Benchmark

This notebook is the Tier 2 deliverable for Module 17. Complete all three implementations, validate correctness, and benchmark scaling. Assessment stubs are at the bottom — implement all five.

---

## MP01: Pairwise Sequence Identity — Three Implementations

**Problem:** Given N=1000 DNA sequences of length L=200, compute the full N×N sequence identity matrix.

$$\text{identity}(a, b) = \frac{\sum_{i=1}^{L} \mathbf{1}[a_i = b_i]}{L}$$

Identity = 1.0 means identical sequences. Identity = 0.0 means every position differs.

**Biological context:** Pairwise sequence identity is used to:
- Define sequence clusters (e.g., 90% identity threshold for redundancy reduction)
- Compute evolutionary distance matrices for phylogenetics
- Filter near-duplicate sequences in databases (UniRef90, CD-HIT)

---

## 1. Data generation

In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt
import pandas as pd

# Fix seed for reproducibility — ALL results in this notebook must be reproducible
np.random.seed(42)
rng = np.random.default_rng(42)

# Check for Numba
try:
    import numba
    NUMBA_AVAILABLE = True
    print(f"Numba available: {numba.__version__}")
except ImportError:
    NUMBA_AVAILABLE = False
    print("Numba not available — Implementation 3 will be skipped")

# --- Data generation ---
N_FULL = 1000  # full benchmark size
L = 200        # sequence length
N_TEST = 20    # small size for correctness validation

# Sequences encoded as integers: A=0, T=1, C=2, G=3
seqs_full = rng.integers(0, 4, size=(N_FULL, L), dtype=np.int8)
seqs_test = seqs_full[:N_TEST]  # small subset for correctness checking

print(f"Full dataset: {seqs_full.shape}  ({seqs_full.nbytes / 1e6:.2f} MB)")
print(f"Test dataset: {seqs_test.shape}")
print(f"Expected full identity matrix size: {N_FULL}×{N_FULL} float32 = {N_FULL**2*4/1e6:.1f} MB")

## 2. Implementation 1 — Python loop (O(N²L) baseline)

In [ ]:
def pairwise_identity_python(seqs):
    """
    Pairwise sequence identity matrix — pure Python double loop.
    
    Complexity: O(N²L) — 200 million comparisons for N=1000, L=200.
    This is the baseline: correct but slow.
    
    Parameters
    ----------
    seqs : np.ndarray, shape (N, L), dtype int8
        Integer-encoded sequences.
    
    Returns
    -------
    np.ndarray, shape (N, N), dtype float32
        Identity matrix where result[i,j] = fraction of matching positions.
    """
    N = len(seqs)
    result = np.zeros((N, N), dtype=np.float32)
    
    for i in range(N):
        result[i, i] = 1.0  # diagonal: sequence is identical to itself
        for j in range(i + 1, N):
            # Count matching positions
            matches = int(np.sum(seqs[i] == seqs[j]))
            identity = matches / L
            result[i, j] = result[j, i] = identity
    
    return result

# Correctness check on test set
t0 = time.perf_counter()
ID_python = pairwise_identity_python(seqs_test)
t_python_test = time.perf_counter() - t0

print(f"Python loop (N={N_TEST}): {t_python_test:.4f} s")
print(f"Sample identity matrix (5×5 corner):\n{ID_python[:5,:5].round(3)}")
print(f"Diagonal = 1.0: {np.all(np.diag(ID_python) == 1.0)}")
print(f"Symmetric: {np.allclose(ID_python, ID_python.T)}")

## 3. Implementation 2 — NumPy broadcasting

In [ ]:
def pairwise_identity_numpy(seqs):
    """
    Pairwise sequence identity matrix — NumPy broadcasting.
    
    Key insight: expand seqs to shape (N, 1, L) and (1, N, L).
    The comparison broadcasts to shape (N, N, L) — no Python loop.
    
    Memory: O(N²L) intermediate boolean tensor.
    For N=1000, L=200: 200 million booleans = 200 MB.
    For N=5000+, this approach runs out of memory — use block-wise instead.
    
    Parameters
    ----------
    seqs : np.ndarray, shape (N, L), dtype int8
    
    Returns
    -------
    np.ndarray, shape (N, N), dtype float32
    """
    # Expand dims for broadcasting:
    # seqs[:, np.newaxis, :]  → shape (N, 1, L)
    # seqs[np.newaxis, :, :]  → shape (1, N, L)
    # comparison              → shape (N, N, L) boolean
    matches = (seqs[:, np.newaxis, :] == seqs[np.newaxis, :, :])  # (N, N, L) bool
    return matches.mean(axis=2).astype(np.float32)                 # (N, N) float32

# Correctness check
t0 = time.perf_counter()
ID_numpy = pairwise_identity_numpy(seqs_test)
t_numpy_test = time.perf_counter() - t0

assert np.allclose(ID_python, ID_numpy, atol=1e-5), "NumPy broadcasting disagrees with Python!"
print(f"NumPy (N={N_TEST}): {t_numpy_test:.4f} s")
print(f"Speedup over Python: {t_python_test/t_numpy_test:.1f}x")
print("Correctness vs Python loop: PASSED")

## 4. Implementation 3 — Numba parallel

In [ ]:
if NUMBA_AVAILABLE:
    import numba
    
    @numba.njit(parallel=True)
    def _pairwise_identity_numba_core(seqs):
        """
        Core Numba kernel — compiled to native code.
        
        Uses numba.prange for the outer loop: each row i is computed
        independently on a separate thread.
        
        The inner loops run at C speed: no Python overhead, no GIL.
        """
        N, L = seqs.shape
        result = np.zeros((N, N), dtype=numba.float32)
        
        for i in numba.prange(N):   # parallel outer loop
            result[i, i] = 1.0
            for j in range(i + 1, N):
                matches = numba.int32(0)
                for k in range(L):
                    if seqs[i, k] == seqs[j, k]:
                        matches += 1
                identity = numba.float32(matches) / numba.float32(L)
                result[i, j] = identity
                result[j, i] = identity
        
        return result
    
    def pairwise_identity_numba(seqs):
        """Wrapper to handle the 2D array."""
        return _pairwise_identity_numba_core(seqs)
    
    # Warmup (compilation)
    print("Compiling Numba kernel... (first call)")
    t0 = time.perf_counter()
    _ = pairwise_identity_numba(seqs_test)
    t_compile = time.perf_counter() - t0
    print(f"Compilation + first run: {t_compile:.3f} s")
    
    # Warm timing
    t0 = time.perf_counter()
    ID_numba = pairwise_identity_numba(seqs_test)
    t_numba_test = time.perf_counter() - t0
    
    assert np.allclose(ID_python, ID_numba, atol=1e-5), "Numba disagrees with Python!"
    print(f"Numba parallel (N={N_TEST}): {t_numba_test:.4f} s  (warm)")
    print(f"Speedup over Python: {t_python_test/t_numba_test:.1f}x")
    print("Correctness vs Python loop: PASSED")
else:
    print("Numba not available — skipping Implementation 3")
    print("Install with: pip install numba")

## 5. Scaling benchmark

In [ ]:
# Scaling analysis across N values
# Python loop is skipped for N > 100 (too slow for N=1000)

Ns = [10, 20, 50, 100, 200, 500, 1000]
times_py = {}
times_np = {}
times_nb = {}

for N in Ns:
    seqs_n = seqs_full[:N]
    
    # NumPy (all sizes)
    t0 = time.perf_counter()
    for _ in range(3):
        pairwise_identity_numpy(seqs_n)
    times_np[N] = (time.perf_counter() - t0) / 3
    
    # Python loop (small sizes only)
    if N <= 100:
        t0 = time.perf_counter()
        pairwise_identity_python(seqs_n)
        times_py[N] = time.perf_counter() - t0
    else:
        # Extrapolate from smaller N (quadratic scaling)
        last_N = max(k for k in times_py if k <= N)
        times_py[N] = times_py[last_N] * (N / last_N) ** 2
    
    # Numba parallel (all sizes)
    if NUMBA_AVAILABLE:
        t0 = time.perf_counter()
        for _ in range(3):
            pairwise_identity_numba(seqs_n)
        times_nb[N] = (time.perf_counter() - t0) / 3

# Print results table
print(f"{'N':>6} {'Python (s)':>12} {'NumPy (s)':>12} {'Numba (s)':>12} {'NP/Py':>8} {'NB/Py':>8}")
for N in Ns:
    tp = times_py[N]
    tn = times_np[N]
    tnb = times_nb.get(N, None)
    py_star = '*' if N > 100 else ' '  # * = extrapolated
    nb_str = f"{tnb:.4f}" if tnb is not None else "  N/A  "
    sp_nb = f"{tp/tnb:.0f}x" if tnb is not None else "N/A"
    print(f"{N:>6} {tp:>11.4f}{py_star} {tn:>12.4f} {nb_str:>12} {tp/tn:>7.0f}x {sp_nb:>8}")
print("* = extrapolated from quadratic scaling")

## 6. Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

Ns_list = Ns

# Panel 1: Scaling curves (log-log)
ax = axes[0]
ax.loglog(Ns_list, [times_py[N] for N in Ns_list], 'o-',
          color='#e74c3c', label='Python loop', linewidth=2)
ax.loglog(Ns_list, [times_np[N] for N in Ns_list], 's-',
          color='#2980b9', label='NumPy broadcast', linewidth=2)
if NUMBA_AVAILABLE:
    ax.loglog(Ns_list, [times_nb[N] for N in Ns_list], '^-',
              color='#27ae60', label='Numba parallel', linewidth=2)

# Overlay O(N²) reference line
n_ref = np.array(Ns_list)
ref_scale = times_py[Ns_list[0]] / Ns_list[0]**2
ax.loglog(n_ref, ref_scale * n_ref**2, 'k--', alpha=0.3, label='O(N²) reference')

ax.set_xlabel('Number of sequences (N)', fontsize=11)
ax.set_ylabel('Wall time (s)', fontsize=11)
ax.set_title('Pairwise Identity: Scaling Curves\n(L=200)', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, which='both', alpha=0.3)

# Panel 2: Speedup ratios
ax = axes[1]
np_speedups = [times_py[N] / times_np[N] for N in Ns_list]
ax.semilogx(Ns_list, np_speedups, 's-', color='#2980b9', label='NumPy / Python', linewidth=2)
if NUMBA_AVAILABLE:
    nb_speedups = [times_py[N] / times_nb[N] for N in Ns_list]
    ax.semilogx(Ns_list, nb_speedups, '^-', color='#27ae60', label='Numba / Python', linewidth=2)
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Number of sequences (N)', fontsize=11)
ax.set_ylabel('Speedup over Python loop', fontsize=11)
ax.set_title('Speedup Ratios vs Input Size', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Panel 3: Identity matrix heatmap (N=20 sample)
ax = axes[2]
im = ax.imshow(ID_numpy, cmap='viridis', vmin=0.3, vmax=1.0, aspect='auto')
plt.colorbar(im, ax=ax, label='Sequence identity')
ax.set_title(f'Identity Matrix Heatmap\n(N={N_TEST} random sequences, L={L})', fontweight='bold')
ax.set_xlabel('Sequence index', fontsize=10)
ax.set_ylabel('Sequence index', fontsize=10)
ax.text(0.02, 0.98, f'Mean: {ID_numpy.mean():.3f}',
        transform=ax.transAxes, color='white', fontsize=9, va='top')

plt.tight_layout()
plt.savefig('../datasets/nb12_mini_project.png', dpi=100, bbox_inches='tight')
plt.show()
print("Mini project visualization complete.")

## 7. Reflection

*Write your reflection here after completing the mini project. What does this exercise teach about the cost of Python abstraction? For what values of N does NumPy broadcasting make sense, and when would a block-wise approach be needed? How does this benchmark connect to real tools like CD-HIT or VSEARCH?*

---

## Assessment Stubs

Implement all five functions. Each represents a core competency from Module 17. Do not look at the notebook implementations while working — these are meant to be attempted from memory first.

Test your implementations using the test harness at the bottom of this section.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ASSESSMENT STUBS — implement all five functions
# ─────────────────────────────────────────────────────────────────────────────

def amdahl_speedup(serial_fraction: float, n_cores: int) -> float:
    """
    Q1: Compute the theoretical speedup given Amdahl's Law.
    
    Parameters
    ----------
    serial_fraction : float
        The fraction of the program that must run serially (0 ≤ s ≤ 1).
    n_cores : int
        Number of parallel processing units.
    
    Returns
    -------
    float
        Theoretical maximum speedup.
    
    Examples
    --------
    >>> amdahl_speedup(0.0, 8)     # fully parallel → 8x
    8.0
    >>> amdahl_speedup(0.5, 4)     # 50% serial → max 2x
    1.333...
    >>> amdahl_speedup(1.0, 100)   # fully serial → no speedup
    1.0
    """
    raise NotImplementedError('Q1: Implement Amdahl speedup formula')


def joblib_parallel_gc(sequences: list, n_jobs: int = -1) -> list:
    """
    Q2: Compute GC content for a list of integer-encoded sequences in parallel
    using joblib.
    
    Parameters
    ----------
    sequences : list of np.ndarray (1D, dtype int8)
        Integer-encoded sequences (A=0, T=1, C=2, G=3).
    n_jobs : int
        Number of parallel workers. -1 = all available CPUs.
    
    Returns
    -------
    list of float
        GC content (0.0 to 1.0) for each sequence, in the same order as input.
    
    Requires: from joblib import Parallel, delayed
    """
    raise NotImplementedError('Q2: Implement joblib parallel GC content')


def numba_sliding_gc(seq_array: np.ndarray, window_size: int) -> np.ndarray:
    """
    Q3: Compute sliding window GC content using a Numba JIT-compiled function.
    
    Parameters
    ----------
    seq_array : np.ndarray, shape (L,), dtype int8
        Integer-encoded sequence.
    window_size : int
        Number of bases per window.
    
    Returns
    -------
    np.ndarray, shape (L - window_size + 1,), dtype float64
        GC fraction for each window position.
    
    Note: the inner function must be decorated with @numba.njit.
    If Numba is not available, use the NumPy cumsum fallback.
    """
    raise NotImplementedError('Q3: Implement Numba JIT sliding GC content')


def numpy_pairwise_hamming(seqs_encoded: np.ndarray) -> np.ndarray:
    """
    Q4: Compute the pairwise Hamming distance matrix using NumPy broadcasting.
    
    Parameters
    ----------
    seqs_encoded : np.ndarray, shape (N, L), dtype int8
        Integer-encoded sequences.
    
    Returns
    -------
    np.ndarray, shape (N, N), dtype float32
        Hamming distance matrix (number of mismatches, not fraction).
    
    Hint: use np.newaxis to create broadcastable dimensions.
    No loops allowed — this must be a single vectorized expression.
    """
    raise NotImplementedError('Q4: Implement NumPy broadcasting pairwise Hamming')


def generate_slurm_script(
    job_name: str,
    n_tasks: int,
    mem_gb: int,
    hours: int,
    command: str
) -> str:
    """
    Q5: Generate a SLURM batch script for a job array.
    
    Parameters
    ----------
    job_name : str
        Name for #SBATCH --job-name.
    n_tasks : int
        Number of array tasks (1 to n_tasks).
    mem_gb : int
        Memory per task in GB for #SBATCH --mem.
    hours : int
        Wall time in hours for #SBATCH --time.
    command : str
        The shell command to run for each task.
    
    Returns
    -------
    str
        A complete SLURM batch script string starting with '#!/bin/bash'.
        Must include at minimum:
          #SBATCH --job-name, --array, --mem, --time
          set -euo pipefail
          echo statements for task info
          the command itself
    """
    raise NotImplementedError('Q5: Implement SLURM script generator')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TEST HARNESS — run this cell after implementing the stubs
# ─────────────────────────────────────────────────────────────────────────────

import traceback

def run_tests():
    passed = 0
    total = 5
    
    # Q1: Amdahl
    try:
        assert abs(amdahl_speedup(0.0, 8) - 8.0) < 1e-6, "Fully parallel: expected 8.0"
        assert abs(amdahl_speedup(1.0, 100) - 1.0) < 1e-6, "Fully serial: expected 1.0"
        expected = 1 / (0.5 + 0.5/4)
        assert abs(amdahl_speedup(0.5, 4) - expected) < 1e-6, f"Expected {expected}"
        print("Q1 PASSED: amdahl_speedup")
        passed += 1
    except NotImplementedError:
        print("Q1 NOT IMPLEMENTED")
    except AssertionError as e:
        print(f"Q1 FAILED: {e}")
    except Exception as e:
        print(f"Q1 ERROR: {e}")
    
    # Q2: joblib parallel GC
    try:
        test_seqs = [rng.integers(0, 4, size=100, dtype=np.int8) for _ in range(20)]
        expected_gc = [float(((s == 2) | (s == 3)).mean()) for s in test_seqs]
        result_gc = joblib_parallel_gc(test_seqs, n_jobs=2)
        assert len(result_gc) == 20, f"Expected 20 results, got {len(result_gc)}"
        assert all(abs(r - e) < 1e-6 for r, e in zip(result_gc, expected_gc)), "GC values disagree"
        print("Q2 PASSED: joblib_parallel_gc")
        passed += 1
    except NotImplementedError:
        print("Q2 NOT IMPLEMENTED")
    except AssertionError as e:
        print(f"Q2 FAILED: {e}")
    except Exception as e:
        print(f"Q2 ERROR: {e}")
    
    # Q3: numba sliding GC
    try:
        test_seq = rng.integers(0, 4, size=500, dtype=np.int8)
        w = 50
        result_gc_slide = numba_sliding_gc(test_seq, w)
        # Verify against numpy cumsum
        gc_bool = (test_seq == 2) | (test_seq == 3)
        cs = np.cumsum(gc_bool.astype(np.int32))
        cs = np.concatenate(([0], cs))
        expected_slide = (cs[w:] - cs[:-w]) / w
        assert len(result_gc_slide) == len(expected_slide), "Length mismatch"
        assert np.allclose(result_gc_slide, expected_slide, atol=1e-5), "Values disagree"
        print("Q3 PASSED: numba_sliding_gc")
        passed += 1
    except NotImplementedError:
        print("Q3 NOT IMPLEMENTED")
    except AssertionError as e:
        print(f"Q3 FAILED: {e}")
    except Exception as e:
        print(f"Q3 ERROR: {e}")
    
    # Q4: numpy pairwise hamming
    try:
        test_seqs_hd = rng.integers(0, 4, size=(10, 50), dtype=np.int8)
        result_hd = numpy_pairwise_hamming(test_seqs_hd)
        assert result_hd.shape == (10, 10), f"Expected (10,10), got {result_hd.shape}"
        # Diagonal must be 0
        assert np.all(np.diag(result_hd) == 0), "Diagonal should be 0 (self-distance)"
        # Symmetric
        assert np.allclose(result_hd, result_hd.T), "Matrix not symmetric"
        # Spot check against reference
        ref_d01 = int(np.sum(test_seqs_hd[0] != test_seqs_hd[1]))
        assert abs(result_hd[0, 1] - ref_d01) < 0.5, f"d[0,1] expected {ref_d01}, got {result_hd[0,1]}"
        print("Q4 PASSED: numpy_pairwise_hamming")
        passed += 1
    except NotImplementedError:
        print("Q4 NOT IMPLEMENTED")
    except AssertionError as e:
        print(f"Q4 FAILED: {e}")
    except Exception as e:
        print(f"Q4 ERROR: {e}")
    
    # Q5: SLURM script generator
    try:
        script = generate_slurm_script(
            job_name='test_job',
            n_tasks=50,
            mem_gb=8,
            hours=2,
            command='echo "Task $SLURM_ARRAY_TASK_ID"'
        )
        assert isinstance(script, str), "Script must be a string"
        assert script.startswith('#!/bin/bash'), "Script must start with #!/bin/bash"
        assert '#SBATCH --job-name=test_job' in script, "Missing job name directive"
        assert '#SBATCH --array=1-50' in script, "Missing array directive with n_tasks=50"
        assert '8G' in script or '8g' in script.lower(), "Missing memory directive with 8GB"
        assert '02:00:00' in script, "Missing time directive for 2 hours"
        print("Q5 PASSED: generate_slurm_script")
        passed += 1
    except NotImplementedError:
        print("Q5 NOT IMPLEMENTED")
    except AssertionError as e:
        print(f"Q5 FAILED: {e}")
    except Exception as e:
        print(f"Q5 ERROR: {e}")
    
    print(f"\n{'='*40}")
    print(f"TOTAL: {passed}/{total} assessments passed")
    if passed == total:
        print("Module 17 assessment: COMPLETE")
    else:
        print(f"Implement the remaining {total-passed} function(s) and re-run.")

run_tests()

<!-- REFERENCE IMPLEMENTATIONS (do not read until you have attempted the stubs)

Q1:
    return 1.0 / (serial_fraction + (1.0 - serial_fraction) / n_cores)

Q2:
    from joblib import Parallel, delayed
    def _gc_single(seq):
        return float(((seq == 2) | (seq == 3)).mean())
    return Parallel(n_jobs=n_jobs)(delayed(_gc_single)(s) for s in sequences)

Q3 (with Numba):
    import numba
    @numba.njit
    def _sliding_gc_jit(seq, w):
        n = len(seq)
        result = np.empty(n - w + 1, dtype=numba.float64)
        for i in range(n - w + 1):
            gc = 0
            for j in range(w):
                if seq[i+j] == 2 or seq[i+j] == 3:
                    gc += 1
            result[i] = gc / w
        return result
    return _sliding_gc_jit(seq_array, window_size)

Q3 (fallback):
    gc_bool = (seq_array == 2) | (seq_array == 3)
    cs = np.cumsum(gc_bool.astype(np.int32))
    cs = np.concatenate(([0], cs))
    return (cs[window_size:] - cs[:-window_size]) / window_size

Q4:
    diff = seqs_encoded[:, np.newaxis, :] != seqs_encoded[np.newaxis, :, :]
    return diff.sum(axis=2).astype(np.float32)

Q5:
    return f"""#!/bin/bash
#SBATCH --job-name={job_name}
#SBATCH --output=logs/{job_name}_%A_%a.out
#SBATCH --error=logs/{job_name}_%A_%a.err
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=4
#SBATCH --mem={mem_gb}G
#SBATCH --time={hours:02d}:00:00
#SBATCH --array=1-{n_tasks}

set -euo pipefail
echo "Job: $SLURM_JOB_ID  Task: $SLURM_ARRAY_TASK_ID"
echo "Started: $(date)"

{command}

echo "Finished: $(date)"
"""

-->

---

## Papers referenced

- Lam, S.K. et al. (2015). "Numba: A LLVM-based Python JIT compiler." — re-read after implementing Q3.
- Mölder, F. et al. (2021). "Sustainable data analysis with Snakemake." — Q5 context.

## References

- CD-HIT (sequence clustering tool): https://github.com/weizhongli/cdhit
- VSEARCH (pairwise identity at scale): https://github.com/torognes/vsearch
- NumPy broadcasting: https://numpy.org/doc/stable/user/basics.broadcasting.html

## Future work / open questions

- The pairwise identity matrix at N=50,000 (a typical clustering task) requires a 10 GB float32 matrix. How would you implement a block-wise approach that computes the matrix in chunks, never loading it all at once?
- CD-HIT uses a "greedy incremental clustering" algorithm that avoids computing the full N×N matrix. How does it work? What is the computational complexity?
- Module 18 (Scientific Writing) will ask you to write a one-page methods section describing this benchmark. Start drafting it in your notebook's reflection cell.